<a href="https://colab.research.google.com/github/Rheaxu/story-converter-ai/blob/main/StoryConverter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# CELL 1: INITIALIZATION, FILE ISOLATION, AND AI LAYOUT VISUAL ANALYSIS
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Installs essential system wrappers (PyMuPDF, Edge-TTS, Google-GenAI).
# 2. Configures clean folder destinations ("storybook_images" & "storybook_audios").
# 3. Strips raw images directly out of the PDF pages.
# 4. Instructs Gemini to explicitly separate the title text block from the story
#    body text block on the first page to allow independent, sequential highlighting.
# ==============================================================================

import os
import json
import subprocess
from google.colab import userdata

# @title 🛠️ Step 1: Initialize Workspace & Run AI Analyzer { display-mode: "form" }
PDF_Filename = "45_FindingTheGiantHat.pdf"  # @param {type:"string"}
Story_Title = "45. Finding The Giant Hat"  # @param {type:"string"}
Folder_Safe_Name = "45_FindingTheGiantHat"  # @param {type:"string"}

print("⚙️ Installing processing dependencies (PyMuPDF, edge-tts, google-genai)...")
subprocess.check_call(["pip", "-q", "install", "pymupdf", "edge-tts", "google-genai"])

import fitz  # PyMuPDF
from google import genai
from google.genai import types

# Establish the exact folder structures requested
project_root = Folder_Safe_Name
audio_dir = os.path.join(project_root, "storybook_audios")
images_dir = os.path.join(project_root, "storybook_images")
os.makedirs(audio_dir, exist_ok=True)
os.makedirs(images_dir, exist_ok=True)

# Verify API key placement
try:
    api_key = userdata.get('GEMINI_API_KEY')
    ai_client = genai.Client(api_key=api_key)
except Exception:
    raise ValueError("❌ Missing GEMINI_API_KEY in your Colab Secrets sidebar.")

if not os.path.exists(PDF_Filename):
    raise FileNotFoundError(f"❌ Could not find '{PDF_Filename}'. Please upload your PDF to the Colab sidebar.")

print("📖 Slicing PDF matrix and saving page graphics...")
doc = fitz.open(PDF_Filename)
full_story_text = ""

for page_num in range(len(doc)):
    page = doc[page_num]
    actual_page_idx = page_num + 1
    full_story_text += f"\n--- PDF PAGE {actual_page_idx} ---\n{page.get_text()}"

    # Render and store graphic frames natively
    pix = page.get_pixmap(dpi=150)
    pix.save(os.path.join(images_dir, f"page{actual_page_idx}.png"))

# Prompt Gemini to return the title and story text as separate structural properties
prompt = f"""
You are an advanced children's literature extraction engine. Inspect the following raw book text page by page.
Identify which pages contain vocabulary review items/word banks, and which pages are active narrative content.

CRITICAL POLICY FOR BOOK TITLES:
- Identify the very first narrative story page. Look for a short, distinct phrase that appears prominently on its own line after any copyright information and *before* or *at the beginning of* the main story text, even if it immediately follows an introductory sentence. This phrase should be treated as the main Book Title.
- For this first page ONLY, you must provide a "title_text" property and an "audio_title_file" property alongside the standard fields. Do not combine them into one text dump.
- IMPORTANT: For the first story page, the `display_text` and `speech_script` properties MUST contain ONLY the pure narrative content. After identifying the `title_text`, you MUST then diligently scan and remove ALL occurrences of this exact `title_text` string from both the `display_text` and `speech_script` for that specific first story page. These fields should be pristine, containing no repetition or inclusion of the title.

CRITICAL POLICY FOR WORD BANKS:
- Do not assume there are exactly 2 word banks.
- Check the words inside every discovered word bank. If you find multiple word banks that contain identical sets of words, consolidate them and ONLY keep one copy of that word bank in your final output map.

Return a single valid JSON object following this exact format:
{{
  "word_banks": [
     {{
       "page_number": 1,
       "words": [
          {{"word": "example", "file": "wb1_0.mp3"}},
          {{"word": "hats", "file": "wb1_1.mp3"}}
       ]
     }}
  ],
  "story_pages": [
     {{
       "page_number": 3,
       "is_first_story_page": true,
       "title_text": "Mr. Bee and the Skunk",
       "audio_title_file": "page3_title.mp3",
       "display_text": "\"Hi!\" Mr. Bee said.",
       "speech_script": "Hi! ... Mr. Bee said.",
       "audio_file": "page3_story.mp3",
       "image_file": "page3.png"
     }},
     {{
       "page_number": 4,
       "is_first_story_page": false,
       "display_text": "\"Ahhhh!!!!\" Mr. Bee said.",
       "speech_script": "Ahhhh!!!! ... Mr. Bee said.",
       "audio_file": "page4.mp3",
       "image_file": "page4.png"
     }}
  ]
}}

Raw Content:
{full_story_text}"""

print("🤖 Querying Gemini to dynamically handle separate text segments...")
response = ai_client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
    config=types.GenerateContentConfig(response_mime_type="application/json"),
)

storybook_manifest = json.loads(response.text)
print("✔️ Text structures analyzed and sequential highlight layout compiled successfully!")

# Start of new validation logic
max_pdf_pages = len(doc)
print(f"📖 Validating page numbers in manifest against actual PDF page count ({max_pdf_pages})...")

for entry_type, entries_list in storybook_manifest.items():
    if entry_type in ["story_pages", "word_banks"]:
        for item in entries_list:
            if "page_number" in item:
                current_page_num = item["page_number"]
                if not (1 <= current_page_num <= max_pdf_pages):
                    print(f"⚠️  WARNING: Found {entry_type.replace('_', ' ')} entry with invalid page_number: {current_page_num}. (Expected 1-{max_pdf_pages})")
                    # For now, just logging the warning is sufficient to identify the problem.
                    # A more robust solution might involve correcting the page_number to a valid one
                    # or removing the entry, depending on the desired behavior for invalid data.
# End of new validation logic

print("✔️ All page number validations complete.")

In [ ]:
# @title 🧪 TEST: Check Title page Information

import json

# Assuming storybook_manifest is available in the kernel state
if 'storybook_manifest' in globals():
    first_story_page = next((page for page in storybook_manifest.get('story_pages', []) if page.get('is_first_story_page')), None)

    if first_story_page:
        # Ensure title_text is string and not None, and remove it from display_text and speech_script
        title_text = first_story_page.get('title_text')
        if title_text:
            original_display_text = first_story_page.get('display_text', '')
            original_speech_script = first_story_page.get('speech_script', '')

            # Remove title_text case-insensitively and strip leading/trailing whitespace
            cleaned_display_text = original_display_text.replace(title_text, '').strip()
            cleaned_speech_script = original_speech_script.replace(title_text, '').strip()

            # Update the dictionary for display in the test cell
            first_story_page['display_text_cleaned'] = cleaned_display_text
            first_story_page['speech_script_cleaned'] = cleaned_speech_script
            # Add original for comparison
            first_story_page['display_text_original'] = original_display_text
            first_story_page['speech_script_original'] = original_speech_script

        print(json.dumps(first_story_page, indent=2))
    else:
        print("No first story page found in storybook_manifest.")
else:
    print("storybook_manifest not found. Please run Step 1 again.")

In [ ]:
# ==============================================================================
# CELL 2: VOCAL RENDERING ENGINE
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Loops through the dynamic word banks and narrative steps.
# 2. If it encounters the designated 'is_first_story_page' flag, it renders an
#    isolated voice track exclusively for the title block, then builds the separate
#    body sentence track.
# ==============================================================================

# @title 🎙️ Step 2: Render Audio Assets { display-mode: "form" }
Voice_Model = "en-US-GuyNeural"  # @param ["en-US-GuyNeural", "en-US-JennyNeural", "en-GB-SoniaNeural", "en-AU-WilliamNeural"]
Word_Bank_Speed = "-10%"  # @param ["-0%", "-5%", "-10%", "-15%", "-20%"]
Story_Reading_Speed = "-15%"  # @param ["-0%", "-5%", "-10%", "-15%", "-20%", "-25%"]

import edge_tts

async def process_vocal_pipeline():
    wb_list = storybook_manifest.get("word_banks", [])
    story_list = storybook_manifest.get("story_pages", [])

    print(f"🎙️ Rendering {len(wb_list)} vocabulary word sets...")
    for wb_idx, bank in enumerate(wb_list):
        bank_id = wb_idx + 1
        for word_idx, w_item in enumerate(bank.get("words", [])):
            filename = f"wb{bank_id}_{word_idx}.mp3"
            w_item['file'] = filename
            path = os.path.join(audio_dir, filename)
            comm = edge_tts.Communicate(w_item['word'], Voice_Model, rate=Word_Bank_Speed)
            await comm.save(path)

    print(f"🎙️ Rendering narrative pages...")
    for page in story_list:
        # Check for isolated title narrative flag and ensure title_text and audio_title_file are not None
        if page.get("is_first_story_page", False) and page.get("title_text") and page.get("audio_title_file"):
            title_path = os.path.join(audio_dir, page['audio_title_file'])
            title_comm = edge_tts.Communicate(page['title_text'], Voice_Model, rate=Story_Reading_Speed)
            await title_comm.save(title_path)

        # Standard page/body track rendering
        path = os.path.join(audio_dir, page['audio_file'])
        comm = edge_tts.Communicate(page['speech_script'], Voice_Model, rate=Story_Reading_Speed)
        await comm.save(path)

    print("✔️ Audio processing execution complete! All structural files rendered!")

await process_vocal_pipeline()

In [ ]:
# ==============================================================================
# CELL 3: INTERACTIVE TRANSLATION INPUT PORTAL WITH CUSTOM IMPORT
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Scans active storybook_manifest configurations.
# 2. Provides an option to specify and load translations from an existing JSON file.
# 3. Renders structured form text areas for vocabulary, story title, and page translations.
# 4. Saves, registers, and overrides the target translation JSON on save.
# ==============================================================================

# @title 🌐 Step 3: Interactive Translation Mapper { display-mode: "form" }

import os
import json
from google.colab import output
import ipywidgets as widgets
from IPython.display import display

wb_list = storybook_manifest.get("word_banks", [])
story_list = storybook_manifest.get("story_pages", [])

default_file_path = "translations_map.json"
existing_translations = {}

# Widgets for custom path loading
import_path_txt = widgets.Text(
    value=default_file_path,
    placeholder="e.g., translations_map.json",
    description="Load Path:",
    layout=widgets.Layout(width='60%')
)

load_button = widgets.Button(
    description="စ Import from JSON",
    button_style='info',
    layout=widgets.Layout(width='30%')
)

import_status = widgets.Label(value="")

# Load initial/default file if it exists
if os.path.exists(default_file_path):
    try:
        with open(default_file_path, "r", encoding="utf-8") as f:
            existing_translations = json.load(f)
        import_status.value = f"Loaded default '{default_file_path}' on initialization."
    except Exception as e:
        import_status.value = f"Error loading default JSON: {e}"

widget_map = {}
widget_list = []

# Story Title Translation Widget
title_header_lbl = widgets.HTML(value="<h3>ဃ Story Title Translation</h3>")
widget_list.append(title_header_lbl)
saved_title_val = existing_translations.get("story_title_translation", "")
title_txt_input = widgets.Textarea(
    value=saved_title_val,
    placeholder=f"Translation for Title: '{Story_Title}'",
    description="Title Translation:",
    layout=widgets.Layout(width='90%', height='50px')
)
widget_map["story_title_translation"] = title_txt_input
widget_list.append(title_txt_input)

# Word Bank widgets
for wb_idx, bank in enumerate(wb_list):
    bank_id = wb_idx + 1
    header_lbl = widgets.HTML(value=f"<h3>ဂ Word Bank {bank_id} Translations</h3>")
    widget_list.append(header_lbl)
    for idx, w in enumerate(bank.get("words", [])):
        word_key = f"wb_{bank_id}_{idx}"
        saved_val = existing_translations.get(word_key, "")
        txt_input = widgets.Textarea(
            value=saved_val,
            placeholder=f"Translation for '{w['word']}'",
            description=f"{w['word']}:",
            layout=widgets.Layout(width='90%', height='50px')
        )
        widget_map[word_key] = txt_input
        widget_list.append(txt_input)

# Story Page widgets
header_lbl = widgets.HTML(value="<h3>ခ Story Page Translations</h3>")
widget_list.append(header_lbl)
for page in story_list:
    p_num = page["page_number"]
    page_key = f"page_{p_num}"
    saved_val = existing_translations.get(page_key, "")
    txt_input = widgets.Textarea(
        value=saved_val,
        placeholder=f"Translation for Page {p_num} body text...",
        description=f"Page {p_num}:",
        layout=widgets.Layout(width='90%', height='80px')
    )
    widget_map[page_key] = txt_input
    widget_list.append(txt_input)

# Import callback logic
def on_load_clicked(b):
    target_path = import_path_txt.value.strip()
    if not target_path:
        import_status.value = "❌ Path cannot be empty!"
        return
    if not os.path.exists(target_path):
        import_status.value = f"❌ File '{target_path}' not found."
        return
    try:
        with open(target_path, "r", encoding="utf-8") as f:
            loaded_data = json.load(f)
        for key, text_area in widget_map.items():
            text_area.value = loaded_data.get(key, "")
        import_status.value = f"✅ Successfully imported translations from '{target_path}'!"
    except Exception as e:
        import_status.value = f"❌ Failed to parse '{target_path}': {e}"

load_button.on_click(on_load_clicked)

save_button = widgets.Button(
    description="င Save and Register Translations",
    button_style='success',
    layout=widgets.Layout(width='40%', height='40px')
)
status_lbl = widgets.Label(value="")

def on_save_clicked(b):
    global Translation_Data_JSON
    temp_map = {}
    for key, text_area in widget_map.items():
        temp_map[key] = text_area.value

    # Register dynamically to global workspace variable
    Translation_Data_JSON = json.dumps(temp_map)

    # Use custom path if valid, otherwise fallback to default
    save_path = import_path_txt.value.strip()
    if not save_path:
        save_path = default_file_path

    try:
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(temp_map, f, ensure_ascii=False, indent=4)
        status_lbl.value = f"✅ Saved to '{save_path}' & registered in memory successfully!"
    except Exception as e:
        status_lbl.value = f"❌ Registered in memory, but failed to write file: {e}"

save_button.on_click(on_save_clicked)

# Package entire UI into structured layout
setup_row = widgets.HBox([import_path_txt, load_button])
container = widgets.VBox([setup_row, import_status, widgets.HTML("<hr>")] + widget_list + [widgets.HTML("<hr>"), save_button, status_lbl])
display(container)

In [ ]:
# ==============================================================================
# CELL 4.1: COMPILE PUBLIC WEB APP (COMPLETE & ENHANCED STYLE ISOLATION)
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Renders public reading platform with toggleable translation blocks outside
#    of original, interactive voice elements.
# 2. Includes dynamic title translations placed neatly under the first story page title.
# 3. Automatically bundles compiled index files with media directories.
# ==============================================================================

# @title 📦 Step 4.1: Compile Public Web App & Download Distribution Package { display-mode: "form" }

import zipfile
import json
import os
from google.colab import files

wb_list = storybook_manifest.get("word_banks", [])
story_list = storybook_manifest.get("story_pages", [])

try:
    translations = json.loads(Translation_Data_JSON)
except Exception:
    translations = {}

# Retrieve story title translation
title_translation = translations.get("story_title_translation", "").strip()
title_tr_html = f'<div class="translation-box"><button class="tr-btn" onclick="event.stopPropagation(); toggleTranslation(this);">🌐</button><div class="tr-content" style="display:none;">{title_translation}</div></div>' if title_translation else ''

# Build dynamic vocabulary sheets with translations
word_banks_html_stack = ""
for wb_idx, bank in enumerate(wb_list):
    bank_id = wb_idx + 1
    items_html = ""
    for idx, w in enumerate(bank.get("words", [])):
        word_key = f"wb_{bank_id}_{idx}"
        tr_val = translations.get(word_key, "").strip()
        tr_html = f'<div class="translation-box"><button class="tr-btn" onclick="event.stopPropagation(); toggleTranslation(this);">🌐</button><div class="tr-content" style="display:none;">{tr_val}</div></div>' if tr_val else ''

        items_html += f"""
        <div class="word-block-wrapper">
            <div class="word-item" onclick="playAudioFile(this, '{w["file"]}')">
                <div class="word-main">{w["word"]}</div>
            </div>
            {tr_html}
        </div>"""

    word_banks_html_stack += f"""
    <div class="page-card">
        <div class="page-number">Word Bank {bank_id}</div>
        <p style="margin:0 0 10px 0; font-weight:bold; color:#777;">Vocabulary Review:</p>
        <div class="word-bank">{items_html}</div>
    </div>"""

# Build story pages layout columns with translations
story_pages_html_stack = ""
for page in story_list:
    p_num = page["page_number"]
    page_key = f"page_{p_num}"
    tr_val = translations.get(page_key, "").strip()
    tr_html = f'<div class="translation-box"><button class="tr-btn" onclick="event.stopPropagation(); toggleTranslation(this);">🌐</button><div class="tr-content" style="display:none;">{tr_val}</div></div>' if tr_val else ''

    if page.get("is_first_story_page", False):
        text_panel_content = f"""
                <div style="width: 100%;">
                    <div class="story-block-wrapper" style="margin-bottom: 15px; border-bottom: 2px dashed #ccc; padding-bottom: 10px;">
                        <div id="first-page-title" class="story-text" style="font-size: 2rem; font-weight: bold; padding: 10px; margin: 0;"
                             onclick="playSequentialFirstPage('storybook_audios/{page['audio_title_file']}', 'storybook_audios/{page['audio_file']}')">
                            {page['title_text']}
                        </div>
                        {title_tr_html}
                    </div>
                    <div class="story-block-wrapper">
                        <div id="first-page-body" class="story-text" onclick="playAudioFile(this, '{page['audio_file']}')">
                            <div class="story-main">{page['display_text']}</div>
                        </div>
                        {tr_html}
                    </div>
                </div>"""
    else:
        text_panel_content = f"""
                <div class="story-block-wrapper">
                    <div class="story-text" onclick="playAudioFile(this, '{page['audio_file']}')">
                        <div class="story-main">{page['display_text']}</div>
                    </div>
                    {tr_html}
                </div>"""

    story_pages_html_stack += f"""
    <div class="page-card">
        <div class="page-number">Page {page['page_number']}</div>
        <div class="story-layout">
            <div class="illustration-side">
                <img src="storybook_images/{page['image_file']}" alt="Page {page['page_number']}" class="story-image">
            </div>
            <div class="text-side">
                {text_panel_content}
            </div>
        </div>
    </div>"""

complete_html_payload = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{Story_Title}</title>
    <style>
        :root {{ --primary: #FFD700; --secondary: #333333; --background: #eef2f5; --card-bg: #ffffff; --text-color: #2c3e50; --interactive-hover: #eaf5ff; --interactive-active: #c6e4ff; }}
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: var(--background); color: var(--text-color); margin: 0; padding: 20px; display: flex; flex-direction: column; align-items: center; }}
        .container {{ max-width: 850px; width: 100%; }}
        header {{ text-align: center; margin-bottom: 25px; background: linear-gradient(135deg, var(--primary), #ffb700); padding: 20px; border-radius: 16px; box-shadow: 0 4px 15px rgba(0,0,0,0.08); }}
        header h1 {{ margin: 0; font-size: 2.2rem; color: var(--secondary); }}
        header p {{ margin: 5px 0 0 0; font-size: 1rem; color: #444; font-weight: 500; }}
        .audio-hint {{ display: flex; align-items: center; justify-content: center; gap: 8px; margin-bottom: 25px; font-size: 0.95rem; color: #555; font-weight: 500; }}
        .audio-hint svg {{ width: 20px; height: 20px; fill: currentColor; }}
        .page-card {{ background-color: var(--card-bg); border-radius: 16px; box-shadow: 0 6px 20px rgba(0,0,0,0.05); padding: 25px; margin-bottom: 30px; position: relative; box-sizing: border-box; }}
        .story-layout {{ display: flex; gap: 25px; align-items: center; margin-top: 10px; }}
        .illustration-side {{ flex: 0 0 55%; max-width: 55%; }}
        .story-image {{ width: 100%; height: auto; display: block; border-radius: 12px; border: 3px solid var(--secondary); box-shadow: 0 4px 10px rgba(0,0,0,0.08); background-color: #fafafa; }}
        .text-side {{ flex: 1; display: flex; align-items: center; }}
        .page-number {{ position: absolute; top: 15px; right: 20px; font-size: 0.8rem; font-weight: bold; color: #bbb; text-transform: uppercase; letter-spacing: 1px; }}

        /* Structured Content Wrappers */
        .word-bank {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(110px, 1fr)); gap: 15px; margin-top: 10px; }}
        .word-block-wrapper {{ display: flex; flex-direction: column; gap: 6px; align-items: stretch; }}
        .story-block-wrapper {{ display: flex; flex-direction: column; gap: 10px; width: 100%; }}

        .word-item {{ background-color: #f8fafc; border: 2px dashed #cbd5e1; border-radius: 10px; padding: 12px 6px; text-align: center; font-size: 1.15rem; font-weight: 600; cursor: pointer; user-select: none; transition: all 0.2s; display: flex; flex-direction: column; align-items: center; gap: 4px; position: relative; }}
        .word-item:hover {{ background-color: #f0fdf4; border-color: #4ade80; transform: scale(1.03); }}
        .story-text {{ font-size: 1.65rem; line-height: 2.4rem; font-weight: 500; color: #1a1a1a; padding: 15px; border-radius: 10px; cursor: pointer; transition: all 0.2s; width: 100%; box-sizing: border-box; border: 2px solid transparent; display: flex; flex-direction: column; gap: 8px; position: relative; }}
        .story-text:hover {{ background-color: var(--interactive-hover); color: #0066cc; border-color: #bae1ff; }}
        .speaking {{ background-color: var(--interactive-active) !important; color: #004499 !important; border-color: #0066cc !important; }}

        /* Separate Box Isolation for Translations */
        .translation-box {{ background-color: #f0fdf4; border: 1px solid #bbf7d0; border-radius: 8px; padding: 10px; display: flex; flex-direction: column; gap: 5px; text-align: left; }}
        .tr-btn {{ background: none; border: none; font-size: 1.3rem; cursor: pointer; outline: none; padding: 0; align-self: flex-start; line-height: 1; transition: transform 0.2s; }}
        .tr-btn:hover {{ transform: scale(1.2); }}
        .tr-content {{ margin-top: 5px; font-size: 1.25rem; font-weight: bold; color: #15803d; border-top: 1px dashed #bbf7d0; padding-top: 6px; }}
        @media(max-width: 768px) {{ .story-layout {{ flex-direction: column; }} .illustration-side {{ flex: 0 0 100%; max-width: 100%; }} }}
    </style>
</head>
<body>
<div class="container">
    <header>
        <h1>{Story_Title}</h1>
        <h3>by Dr. Yvonne Crawford</h3>
        <p>Interactive Read-Along Experience</p>
    </header>
    <div class="audio-hint">
        <svg viewBox="0 0 24 24"><path d="M3 9v6h4l5 5V4L7 9H3zm13.5 3c0-1.77-1.02-3.25-2.5-4.01v8.03c1.48-.76 2.5-2.24 2.5-4.02zM14 3.23v2.06c2.89.86 5 3.54 5 6.71s-2.11 5.85-5 6.71v2.06c4.01-.91 7-4.49 7-8.77s-2.99-7-8.77z"/></svg>
        <span>Click elements to play professional voice track!</span>
    </div>
    {word_banks_html_stack}
    {story_pages_html_stack}
</div>
<script>
    let currentAudio = null; let currentElement = null;

    function resetActiveStates() {{
        if (currentAudio) {{ currentAudio.pause(); currentAudio.currentTime = 0; }}
        if (currentElement) {{ currentElement.classList.remove('speaking'); }}
    }}

    function playAudioFile(element, filename) {{
        resetActiveStates();
        currentElement = element;
        currentElement.classList.add('speaking');
        currentAudio = new Audio('storybook_audios/' + filename);
        currentAudio.play().catch(error => {{
            console.log('Playback issue:', error);
            if (currentElement) currentElement.classList.remove('speaking');
        }});
        currentAudio.onended = function () {{
            if (currentElement) {{ currentElement.classList.remove('speaking'); }}
        }};
    }}

    function playSequentialFirstPage(titleAudioSrc, bodyAudioSrc) {{
        resetActiveStates();
        let titleEl = document.getElementById('first-page-title');
        let bodyEl = document.getElementById('first-page-body');
        currentElement = titleEl;
        currentElement.classList.add('speaking');
        currentAudio = new Audio(titleAudioSrc);
        currentAudio.play().catch(e => console.log(e));
        currentAudio.onended = function () {{
            titleEl.classList.remove('speaking');
            setTimeout(function () {{
                currentElement = bodyEl;
                currentElement.classList.add('speaking');
                currentAudio = new Audio(bodyAudioSrc);
                currentAudio.play().catch(e => console.log(e));
                currentAudio.onended = function () {{ bodyEl.classList.remove('speaking'); }};
            }}, 1200);
        }};
    }}

    function toggleTranslation(buttonElement) {{
        const translationBox = buttonElement.nextElementSibling;
        if (translationBox.style.display === "none") {{
            translationBox.style.display = "block";
        }} else {{
            translationBox.style.display = "none";
        }}
    }}
</script>
</body>
</html>"""

with open(os.path.join(project_root, "index.html"), "w", encoding="utf-8") as f:
    f.write(complete_html_payload)

print("ခ Packaging pipeline zip module distribution payload (Public)...")
zip_filename = f"{Folder_Safe_Name}_production_package.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, file_list in os.walk(project_root):
        for file in file_list:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, os.path.dirname(project_root))
            zipf.write(file_path, arcname)

files.download(zip_filename)
print("ထ Complete bundle dispatched directly to public downloads archive!")


In [ ]:
# ==============================================================================
# CELL 4.2: COMPILE PRIVATE WEB APP (COMPLETE & ENHANCED STYLE ISOLATION)
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Renders private SAS reading platform with toggleable translation blocks outside
#    of original, interactive voice elements.
# 2. Displays the translated story title dynamically under the first story page title.
# 3. Implements secure dynamic rewrite engine for Cloud Storage audio assets.
# ==============================================================================

# @title 📦 Step 4.2: Compile Private Web App & Download Distribution Package { display-mode: "form" }

import zipfile
import json
import os
from google.colab import files

wb_list = storybook_manifest.get("word_banks", [])
story_list = storybook_manifest.get("story_pages", [])

try:
    translations = json.loads(Translation_Data_JSON)
except Exception:
    translations = {}

# Retrieve story title translation
title_translation = translations.get("story_title_translation", "").strip()
title_tr_html = f'<div class="translation-box"><button class="tr-btn" onclick="event.stopPropagation(); toggleTranslation(this);">🌐</button><div class="tr-content" style="display:none;">{title_translation}</div></div>' if title_translation else ''

# Dynamic vocabulary rendering
word_banks_html_stack = ""
for wb_idx, bank in enumerate(wb_list):
    bank_id = wb_idx + 1
    items_html = ""
    for idx, w in enumerate(bank.get("words", [])):
        word_key = f"wb_{bank_id}_{idx}"
        tr_val = translations.get(word_key, "").strip()
        tr_html = f'<div class="translation-box"><button class="tr-btn" onclick="event.stopPropagation(); toggleTranslation(this);">🌐</button><div class="tr-content" style="display:none;">{tr_val}</div></div>' if tr_val else ''
        items_html += f"""
        <div class="word-block-wrapper">
            <div class="word-item" onclick="playAudioFile(this, '{w["file"]}')">
                <div class="word-main">{w["word"]}</div>
            </div>
            {tr_html}
        </div>"""
    word_banks_html_stack += f"""
    <div class="page-card">
        <div class="page-number">Word Bank {bank_id}</div>
        <p style="margin:0 0 10px 0; font-weight:bold; color:#777;">Vocabulary Review:</p>
        <div class="word-bank">{items_html}</div>
    </div>"""

# Dynamic page rendering
story_pages_html_stack = ""
for page in story_list:
    p_num = page["page_number"]
    page_key = f"page_{p_num}"
    tr_val = translations.get(page_key, "").strip()
    tr_html = f'<div class="translation-box"><button class="tr-btn" onclick="event.stopPropagation(); toggleTranslation(this);">🌐</button><div class="tr-content" style="display:none;">{tr_val}</div></div>' if tr_val else ''

    if page.get("is_first_story_page", False):
        text_panel_content = f"""
        <div style="width: 100%;">
            <div class="story-block-wrapper" style="margin-bottom: 15px; border-bottom: 2px dashed #ccc; padding-bottom: 10px;">
                <div id="first-page-title" class="story-text" style="font-size: 2rem; font-weight: bold; padding: 10px; margin: 0;"
                     onclick="playSequentialFirstPage('storybook_audios/{page['audio_title_file']}', 'storybook_audios/{page['audio_file']}')">
                    {page['title_text']}
                </div>
                {title_tr_html}
            </div>
            <div class="story-block-wrapper">
                <div id="first-page-body" class="story-text" onclick="playAudioFile(this, '{page['audio_file']}')">
                    <div class="story-main">{page['display_text']}</div>
                </div>
                {tr_html}
            </div>
        </div>"""
    else:
        text_panel_content = f"""
        <div class="story-block-wrapper">
            <div class="story-text" onclick="playAudioFile(this, '{page['audio_file']}')">
                <div class="story-main">{page['display_text']}</div>
            </div>
            {tr_html}
        </div>"""

    story_pages_html_stack += f"""
    <div class="page-card">
        <div class="page-number">Page {page['page_number']}</div>
        <div class="story-layout">
            <div class="illustration-side">
                <img src="storybook_images/{page['image_file']}" alt="Page {page['page_number']}" class="story-image">
            </div>
            <div class="text-side">
                {text_panel_content}
            </div>
        </div>
    </div>"""

complete_html_payload = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{Story_Title}</title>
    <style>
        :root {{ --primary: #FFD700; --secondary: #333333; --background: #eef2f5; --card-bg: #ffffff; --text-color: #2c3e50; --interactive-hover: #eaf5ff; --interactive-active: #c6e4ff; }}
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: var(--background); color: var(--text-color); margin: 0; padding: 20px; display: flex; flex-direction: column; align-items: center; }}
        .container {{ max-width: 850px; width: 100%; }}
        header {{ text-align: center; margin-bottom: 25px; background: linear-gradient(135deg, var(--primary), #ffb700); padding: 20px; border-radius: 16px; box-shadow: 0 4px 15px rgba(0,0,0,0.08); }}
        header h1 {{ margin: 0; font-size: 2.2rem; color: var(--secondary); }}
        header p {{ margin: 5px 0 0 0; font-size: 1rem; color: #444; font-weight: 500; }}
        .audio-hint {{ display: flex; align-items: center; justify-content: center; gap: 8px; margin-bottom: 25px; font-size: 0.95rem; color: #555; font-weight: 500; }}
        .audio-hint svg {{ width: 20px; height: 20px; fill: currentColor; }}
        .page-card {{ background-color: var(--card-bg); border-radius: 16px; box-shadow: 0 6px 20px rgba(0,0,0,0.05); padding: 25px; margin-bottom: 30px; position: relative; box-sizing: border-box; }}
        .story-layout {{ display: flex; gap: 25px; align-items: center; margin-top: 10px; }}
        .illustration-side {{ flex: 0 0 55%; max-width: 55%; }}
        .story-image {{ width: 100%; height: auto; display: block; border-radius: 12px; border: 3px solid var(--secondary); box-shadow: 0 4px 10px rgba(0,0,0,0.08); background-color: #fafafa; }}
        .text-side {{ flex: 1; display: flex; align-items: center; }}
        .page-number {{ position: absolute; top: 15px; right: 20px; font-size: 0.8rem; font-weight: bold; color: #bbb; text-transform: uppercase; letter-spacing: 1px; }}

        /* Structured Content Wrappers */
        .word-bank {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(110px, 1fr)); gap: 15px; margin-top: 10px; }}
        .word-block-wrapper {{ display: flex; flex-direction: column; gap: 6px; align-items: stretch; }}
        .story-block-wrapper {{ display: flex; flex-direction: column; gap: 10px; width: 100%; }}

        .word-item {{ background-color: #f8fafc; border: 2px dashed #cbd5e1; border-radius: 10px; padding: 12px 6px; text-align: center; font-size: 1.15rem; font-weight: 600; cursor: pointer; user-select: none; transition: all 0.2s; display: flex; flex-direction: column; align-items: center; gap: 4px; position: relative; }}
        .word-item:hover {{ background-color: #f0fdf4; border-color: #4ade80; transform: scale(1.03); }}
        .story-text {{ font-size: 1.65rem; line-height: 2.4rem; font-weight: 500; color: #1a1a1a; padding: 15px; border-radius: 10px; cursor: pointer; transition: all 0.2s; width: 100%; box-sizing: border-box; border: 2px solid transparent; display: flex; flex-direction: column; gap: 8px; position: relative; }}
        .story-text:hover {{ background-color: var(--interactive-hover); color: #0066cc; border-color: #bae1ff; }}
        .speaking {{ background-color: var(--interactive-active) !important; color: #004499 !important; border-color: #0066cc !important; }}

        /* Separate Box Isolation for Translations */
        .translation-box {{ background-color: #f0fdf4; border: 1px solid #bbf7d0; border-radius: 8px; padding: 10px; display: flex; flex-direction: column; gap: 5px; text-align: left; }}
        .tr-btn {{ background: none; border: none; font-size: 1.3rem; cursor: pointer; outline: none; padding: 0; align-self: flex-start; line-height: 1; transition: transform 0.2s; }}
        .tr-btn:hover {{ transform: scale(1.2); }}
        .tr-content {{ margin-top: 5px; font-size: 1.25rem; font-weight: bold; color: #15803d; border-top: 1px dashed #bbf7d0; padding-top: 6px; }}
        @media(max-width: 768px) {{ .story-layout {{ flex-direction: column; }} .illustration-side {{ flex: 0 0 100%; max-width: 100%; }} }}
    </style>
</head>
<body>
<div class="container">
    <header>
        <h1>{Story_Title}</h1>
        <h3>by Dr. Yvonne Crawford</h3>
        <p>Interactive Read-Along Experience</p>
    </header>
    <div class="audio-hint">
        <svg viewBox="0 0 24 24"><path d="M3 9v6h4l5 5V4L7 9H3zm13.5 3c0-1.77-1.02-3.25-2.5-4.01v8.03c1.48-.76 2.5-2.24 2.5-4.02zM14 3.23v2.06c2.89.86 5 3.54 5 6.71s-2.11 5.85-5 6.71v2.06c4.01-.91 7-4.49 7-8.77s-2.99-7-8.77z"/></svg>
        <span>Click elements to play professional voice track!</span>
    </div>
    {word_banks_html_stack}
    {story_pages_html_stack}
</div>
<script>
    let currentAudio = null; let currentElement = null;

    function resetActiveStates() {{
        if (currentAudio) {{ currentAudio.pause(); currentAudio.currentTime = 0; }}
        if (currentElement) {{ currentElement.classList.remove('speaking'); }}
    }}

    function playAudioFile(element, filename) {{
        resetActiveStates();
        currentElement = element;
        currentElement.classList.add('speaking');
        currentAudio = new Audio(filename);
        currentAudio.play().catch(error => {{
            console.log('Playback issue:', error);
            if (currentElement) currentElement.classList.remove('speaking');
        }});
        currentAudio.onended = function () {{
            if (currentElement) {{ currentElement.classList.remove('speaking'); }}
        }};
    }}

    function playSequentialFirstPage(titleAudioSrc, bodyAudioSrc) {{
        resetActiveStates();
        let titleEl = document.getElementById('first-page-title');
        let bodyEl = document.getElementById('first-page-body');
        currentElement = titleEl;
        currentElement.classList.add('speaking');
        currentAudio = new Audio(titleAudioSrc);
        currentAudio.play().catch(e => console.log(e));
        currentAudio.onended = function () {{
            titleEl.classList.remove('speaking');
            setTimeout(function () {{
                currentElement = bodyEl;
                currentElement.classList.add('speaking');
                currentAudio = new Audio(bodyAudioSrc);
                currentAudio.play().catch(e => console.log(e));
                currentAudio.onended = function () {{ bodyEl.classList.remove('speaking'); }};
            }}, 1200);
        }};
    }}

    function toggleTranslation(buttonElement) {{
        const translationBox = buttonElement.nextElementSibling;
        if (translationBox.style.display === "none") {{
            translationBox.style.display = "block";
        }} else {{
            translationBox.style.display = "none";
        }}
    }}

    (function () {{
        const sas = window.location.search;
        if (!sas) return;
        const parts = window.location.pathname.split("/");
        const containerIndex = parts.indexOf("yvonnereadingprivate");
        const folderPath = parts.slice(containerIndex + 1, -1).join("/");
        const BASE = window.location.origin + "/yvonnereadingprivate/" + folderPath + "/";

        function resolveAsset(src) {{
            if (!src) return src;
            if (src.startsWith("http")) return src;
            src = src.replace(/^\/+/, "");
            if (!src.startsWith("storybook_images/") && !src.startsWith("storybook_audios/")) {{
                if (src.endsWith(".mp3")) {{ src = "storybook_audios/" + src; }} else {{ src = "storybook_images/" + src; }}
            }}
            return BASE + src + sas;
        }}

        document.querySelectorAll("img").forEach(img => {{
            const src = img.getAttribute("src");
            img.src = resolveAsset(src);
        }});

        const originalAudio = window.Audio;
        window.Audio = function (src) {{
            return new originalAudio(resolveAsset(src));
        }};
    }})();
</script>
</body>
</html>"""

with open(os.path.join(project_root, "index.html"), "w", encoding="utf-8") as f:
    f.write(complete_html_payload)

print("ခ Packaging pipeline zip module distribution payload (Private)...")
zip_filename = f"{Folder_Safe_Name}_private_production_package.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, file_list in os.walk(project_root):
        for file in file_list:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, os.path.dirname(project_root))
            zipf.write(file_path, arcname)

files.download(zip_filename)
print("ထ Complete bundle dispatched directly to private downloads archive!")


In [ ]:
# ==============================================================================
# CELL 5a: LOGIN PAGE CREATION & DISTRIBUTION PACKAGER
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Creates an 'index.html' file for a login page.
# 2. Places this file in a '<Folder_Safe_Name>' directory structure.
# 3. Packages the login page into a downloadable zip file.
# ==============================================================================

# @title 📦 Step 5a: Create Login Page & Download Package { display-mode: "form" }

import zipfile
from google.colab import files
import os

# Create the folder directly under the current working directory
login_app_root = Folder_Safe_Name + "_original"
os.makedirs(login_app_root, exist_ok=True)

login_html_payload = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Reading Access</title>
    <style>
        :root {{ --primary: #FFD700; --secondary: #333333; --background: #eef2f5; --card-bg: #ffffff; --text-color: #2c3e50; --interactive-hover: #eaf5ff; --interactive-active: #c6e4ff; }}
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: var(--background); color: var(--text-color); margin: 0; padding: 20px; display: flex; flex-direction: column; align-items: center; min-height: 100vh; justify-content: center; }} /* Added min-height and justify-content */
        .container {{ max-width: 400px; width: 100%; }} /* Reduced max-width for login */
        header {{ text-align: center; margin-bottom: 25px; background: linear-gradient(135deg, var(--primary), #ffb700); padding: 20px; border-radius: 16px; box-shadow: 0 4px 15px rgba(0,0,0,0.08); }}
        header h1 {{ margin: 0; font-size: 2.2rem; color: var(--secondary); }}
        header p {{ margin: 5px 0 0 0; font-size: 1rem; color: #444; font-weight: 500; }}

        .login-card {{ background-color: var(--card-bg); border-radius: 16px; box-shadow: 0 6px 20px rgba(0,0,0,0.05); padding: 30px; position: relative; box-sizing: border-box; }}
        .form-group {{ margin-bottom: 20px; }}
        label {{ display: block; margin-bottom: 8px; font-weight: 600; color: var(--text-color); }}
        .password-input-wrapper {{ display: flex; border: 1px solid #ccc; border-radius: 8px; overflow: hidden; }}
        .password-input-wrapper input {{ flex-grow: 1; border: none; padding: 12px; font-size: 1rem; outline: none; }}
        .password-input-wrapper button {{ background-color: #eee; border: none; padding: 0 15px; cursor: pointer; font-size: 0.9rem; color: #555; transition: background-color 0.2s; }}
        .password-input-wrapper button:hover {{ background-color: #ddd; }}
        .password-input-wrapper button:disabled {{ background-color: #f5f5f5; cursor: not-allowed; }}


        button.submit-btn {{ background-color: var(--primary); color: var(--secondary); border: none; padding: 12px 20px; border-radius: 8px; font-size: 1.1rem; font-weight: bold; cursor: pointer; transition: background-color 0.3s ease; width: 100%; }}
        button.submit-btn:hover {{ background-color: #ffc107; }}
        button.submit-btn:disabled {{ background-color: #ccc; cursor: not-allowed; color: #666; }}

        p#error {{ color: red; text-align: center; margin-top: 15px; font-size: 0.9rem; }}

        .loading-overlay {{
            position: fixed; top: 0; left: 0; width: 100%; height: 100%;
            background-color: rgba(255, 255, 255, 0.8);
            display: none; /* Hidden by default */
            align-items: center; justify-content: center;
            z-index: 1000;
        }}
        .loading-spinner {{
            border: 4px solid #f3f3f3; border-top: 4px solid var(--primary);
            border-radius: 50%; width: 40px; height: 40px;
            animation: spin 1s linear infinite;
        }}
        @keyframes spin {{ 0% {{ transform: rotate(0deg); }} 100% {{ transform: rotate(360deg); }} }}
    </style>
</head>
<body>
<div class="container">
    <header>
        <h1>Log In</h1>
        <p>Enter your password to proceed.</p>
    </header>
    <div class="login-card">
        <div class="form-group">
            <label for="pw">Password</label>
            <div class="password-input-wrapper">
                <input type="password" id="pw" placeholder="Enter password" autocomplete="off" />
                <button type="button" id="togglePassword" onclick="togglePasswordVisibility()">Show</button>
            </div>
        </div>
        <button type="button" id="submitBtn" class="submit-btn" onclick="login()">Submit</button>
        <p id="error"></p>
    </div>
</div>
<div id="loadingOverlay" class="loading-overlay">
    <div class="loading-spinner"></div>
</div>
<script>
    const functionUrl = "https://children-english-reading-password-api-e0cqgjbaf2fmbtaj.eastasia-01.azurewebsites.net/api/GetSasUrl";

    function togglePasswordVisibility() {{
        const passwordInput = document.getElementById('pw');
        const toggleButton = document.getElementById('togglePassword');
        if (passwordInput.type === 'password') {{
            passwordInput.type = 'text';
            toggleButton.textContent = 'Hide';
        }} else {{
            passwordInput.type = 'password';
            toggleButton.textContent = 'Show';
        }}
    }}

    async function login() {{
        const password = document.getElementById("pw").value;
        const errorParagraph = document.getElementById("error");
        const submitButton = document.getElementById("submitBtn");
        const passwordInput = document.getElementById("pw");
        const toggleButton = document.getElementById("togglePassword");
        const loadingOverlay = document.getElementById("loadingOverlay");

        errorParagraph.textContent = ''; // Clear previous errors

        // Disable inputs/buttons and show loader
        passwordInput.disabled = true;
        submitButton.disabled = true;
        toggleButton.disabled = true;
        loadingOverlay.style.display = 'flex';

        try {{
            // Update logic to match the uglified version, capturing the full relative path
            const parts = window.location.pathname.split("/").filter(p=>p!=="");
            const relativePath = parts.join("/");

            const response = await fetch(functionUrl, {{
                method: "POST",
                headers: {{ "Content-Type": "application/json" }},
                body: JSON.stringify({{
                    password,
                    path: relativePath
                }})
            }});

            const data = await response.json();

            if (!response.ok) {{
                errorParagraph.textContent = data.error || "An unexpected error occurred.";
            }} else {{
                window.location.href = data.url;
            }}
        }} catch (e) {{
            console.error("Login failed:", e);
            errorParagraph.textContent = "Network error or invalid response.";
        }} finally {{
            // Re-enable inputs/buttons and hide loader
            passwordInput.disabled = false;
            submitButton.disabled = false;
            toggleButton.disabled = false;
            loadingOverlay.style.display = 'none';
        }}
    }}
</script>
</body>
</html>"""

with open(os.path.join(login_app_root, "index.html"), "w", encoding="utf-8") as f:
    f.write(login_html_payload)

print("📦 Packaging login page distribution payload...")
zip_filename = f"{Folder_Safe_Name}_original_login_page.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # We want Folder_Safe_Name/index.html inside the zip
    file_path = os.path.join(login_app_root, "index.html")
    # Set arcname to include the Folder_Safe_Name as the top-level directory in the zip
    arcname_in_zip = os.path.join(Folder_Safe_Name + "_original", "index.html")
    zipf.write(file_path, arcname=arcname_in_zip)

files.download(zip_filename)
print("🚀 Login page bundle dispatched directly to downloads archive!")

In [ ]:
# ==============================================================================
# CELL 5 (Uglified): LOGIN PAGE CREATION & DISTRIBUTION PACKAGER
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Creates an 'index.html' file for a login page with uglified JavaScript.
# 2. Places this file in a '<Folder_Safe_Name>' directory structure.
# 3. Packages the uglified login page into a downloadable zip file.
# ==============================================================================

# @title 📦 Step 5b (Uglified & Minified): Create Login Page & Download Package { display-mode: "form" }
import subprocess
print("⚙️ Installing processing dependencies (jsmin, htmlmin)...")
subprocess.check_call(["pip", "-q", "install", "jsmin", "htmlmin"])
from jsmin import jsmin
import htmlmin

import zipfile
from google.colab import files
import os

# The user provided an already uglified and minified HTML. We will directly use this as the final payload.
login_html_payload_uglified = """<!DOCTYPE html><html lang=en><head><meta charset=UTF-8><meta name=viewport content="width=device-width, initial-scale=1.0"><title>Reading Access</title><style>
        :root { --primary: #FFD700; --secondary: #333333; --background: #eef2f5; --card-bg: #ffffff; --text-color: #2c3e50; --interactive-hover: #eaf5ff; --interactive-active: #c6e4ff; }
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: var(--background); color: var(--text-color); margin: 0; padding: 20px; display: flex; flex-direction: column; align-items: center; min-height: 100vh; justify-content: center; }
        .container { max-width: 400px; width: 100%; }
        header { text-align: center; margin-bottom: 25px; background: linear-gradient(135deg, var(--primary), #ffb700); padding: 20px; border-radius: 16px; box-shadow: 0 4px 15px rgba(0,0,0,0.08); }
        header h1 { margin: 0; font-size: 2.2rem; color: var(--secondary); }
        header p { margin: 5px 0 0 0; font-size: 1rem; color: #444; font-weight: 500; }

        .login-card { background-color: var(--card-bg); border-radius: 16px; box-shadow: 0 6px 20px rgba(0,0,0,0.05); padding: 30px; position: relative; box-sizing: border-box; }
        .form-group { margin-bottom: 20px; }
        label { display: block; margin-bottom: 8px; font-weight: 600; color: var(--text-color); }
        .password-input-wrapper { display: flex; border: 1px solid #ccc; border-radius: 8px; overflow: hidden; }
        .password-input-wrapper input { flex-grow: 1; border: none; padding: 12px; font-size: 1rem; outline: none; }
        .password-input-wrapper button { background-color: #eee; border: none; padding: 0 15px; cursor: pointer; font-size: 0.9rem; color: #555; transition: background-color 0.2s; }
        .password-input-wrapper button:hover { background-color: #ddd; }
        .password-input-wrapper button:disabled { background-color: #f5f5f5; cursor: not-allowed; }


        button.submit-btn { background-color: var(--primary); color: var(--secondary); border: none; padding: 12px 20px; border-radius: 8px; font-size: 1.1rem; font-weight: bold; cursor: pointer; transition: background-color 0.3s ease; width: 100%; }
        button.submit-btn:hover { background-color: #ffc107; }
        button.submit-btn:disabled { background-color: #ccc; cursor: not-allowed; color: #666; }

        p#error { color: red; text-align: center; margin-top: 15px; font-size: 0.9rem; }

        .loading-overlay {
            position: fixed; top: 0; left: 0; width: 100%; height: 100%;
            background-color: rgba(255, 255, 255, 0.8);
            display: none;
            align-items: center; justify-content: center;
            z-index: 1000;
        }
        .loading-spinner {
            border: 4px solid #f3f3f3; border-top: 4px solid var(--primary);
            border-radius: 50%; width: 40px; height: 40px;
            animation: spin 1s linear infinite;
        }
        @keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }
    </style></head><body><div class=container><header><h1>Log In</h1><p>Enter your password to proceed.</p></header><div class=login-card><div class=form-group><label for=pw>Password</label><div class=password-input-wrapper><input type=password id=pw placeholder="Enter password" autocomplete=off><button type=button id=togglePassword onclick=togglePasswordVisibility()>Show</button></div></div><button type=button id=submitBtn class=submit-btn onclick=login()>Submit</button><p id=error></p></div></div><div id=loadingOverlay class=loading-overlay><div class=loading-spinner></div></div><script>const functionUrl="https://children-english-reading-password-api-e0cqgjbaf2fmbtaj.eastasia-01.azurewebsites.net/api/GetSasUrl";function togglePasswordVisibility(){const passwordInput=document.getElementById('pw');const toggleButton=document.getElementById('togglePassword');if(passwordInput.type==='password'){passwordInput.type='text';toggleButton.textContent='Hide';}else{passwordInput.type='password';toggleButton.textContent='Show';}}
async function login(){const password=document.getElementById("pw").value;const errorParagraph=document.getElementById("error");const submitButton=document.getElementById("submitBtn");const passwordInput=document.getElementById("pw");const toggleButton=document.getElementById("togglePassword");const loadingOverlay=document.getElementById("loadingOverlay");errorParagraph.textContent='';passwordInput.disabled=true;submitButton.disabled=true;toggleButton.disabled=true;loadingOverlay.style.display='flex';try{const parts=window.location.pathname.split("/").filter(p=>p!=="");const relativePath=parts.join("/");const response=await fetch(functionUrl,{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({password,path:relativePath})});const rawText=await response.text();let data;try{data=JSON.parse(rawText);}catch(parseErr){console.error("Non-JSON response:",response.status,rawText);errorParagraph.textContent="Server returned an unexpected response (HTTP "+response.status+"). Check the function logs.";return;}if(!response.ok){errorParagraph.textContent=data.error||"An unexpected error occurred.";}else{window.location.href=data.url;}}catch(e){console.error("Login failed:",e);errorParagraph.textContent="Network error or invalid response.";}finally{passwordInput.disabled=false;submitButton.disabled=false;toggleButton.disabled=false;loadingOverlay.style.display='none';}}</script></body></html>"""

login_app_root_uglified = Folder_Safe_Name
os.makedirs(login_app_root_uglified, exist_ok=True)

with open(os.path.join(login_app_root_uglified, "index.html"), "w", encoding="utf-8") as f:
    f.write(login_html_payload_uglified)

print("📦 Packaging uglified login page distribution payload...")
zip_filename_uglified = f"{Folder_Safe_Name}_uglified_login_page.zip"
with zipfile.ZipFile(zip_filename_uglified, 'w', zipfile.ZIP_DEFLATED) as zipf:
    file_path_uglified = os.path.join(login_app_root_uglified, "index.html")
    arcname_in_zip_uglified = os.path.join(Folder_Safe_Name, "index.html")
    zipf.write(file_path_uglified, arcname=arcname_in_zip_uglified)

files.download(zip_filename_uglified)
print("🚀 Uglified login page bundle dispatched directly to downloads archive!")

The AI's policy for book titles is very strict. It requires the title text to be *completely isolated* from the story sentence on the *first narrative page*. Even if it appears on a new line in the raw text, if it's immediately after a sentence without significant whitespace or a clear structural break recognized by the text extraction, the AI might interpret it as a continuation or a part of the narrative content.

Let's display the `full_story_text` around page 2 to see exactly what Gemini received:

In [ ]:
# @title 🧪 TEST: Check Page 2 content
import re

# Find the relevant section for PDF PAGE 2
match = re.search(r'--- PDF PAGE 2 ---\n(.*?)--- PDF PAGE 3 ---', full_story_text, re.DOTALL)

if match:
    page2_text = match.group(1).strip()
    print(page2_text)
else:
    print("Could not find text for PDF PAGE 2.")


In [ ]:
# ==============================================================================
# CELL 6: WORKSPACE CLEANUP ENGINE
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Provides a graphical form field for you to specify any target directory.
# 2. Checks if that specific folder exists in your current Colab environment.
# 3. Recursively deletes all files, images, audios, and subfolders inside it
#    to cleanly reset your workspace.
# ==============================================================================

# @title 🧹 Step 6: Clear Specified Project Directory { display-mode: "form" }
target_directory = "23_TheRainbow"  # @param {type:"string"}

import shutil
import os

# Clean up any trailing slashes user might type by accident
target_directory = target_directory.strip().rstrip('/')

if target_directory and os.path.exists(target_directory):
    print(f"⚠️ Initializing cleanup sequence for directory: '{target_directory}'...")

    try:
        # Recursively deletes the entire directory tree
        shutil.rmtree(target_directory)
        print(f"🗑️ Clean sweep complete! '{target_directory}' and all of its contents have been deleted.")
    except Exception as e:
        print(f"❌ An error occurred while trying to delete the folder: {e}")

elif not target_directory:
    print("❌ Error: Please specify a valid folder name in the target_directory box.")
else:
    print(f"ℹ️ The folder '{target_directory}' does not exist or has already been cleared.")